In [ ]:
import datetime as dt
import pandas as pd
from gs_quant.instrument import EqOption
from gs_quant.backtests.triggers import (
    MktTriggerRequirements,
    MktTrigger,
    TriggerDirection,
    AggregateTrigger,
    AggregateTriggerRequirements,
    AggType,
    TradeCountTriggerRequirements,
)
from gs_quant.backtests.actions import AddScaledTradeAction, ScalingActionType, ExitAllPositionsAction
from gs_quant.backtests.generic_engine import GenericEngine
from gs_quant.backtests.strategy import Strategy
from gs_quant.backtests.data_sources import GenericDataSource, MissingDataStrategy
from gs_quant.data import Dataset
from gs_quant.risk import EqVega

In [ ]:
# Initialize session
from gs_quant.session import GsSession

GsSession.use(client_id=None, client_secret=None, scopes=('run_analytics'))

#### This notebook shows to do partial/ gradual buys of instruments based on signal.  

E.g. “backtest buying a VIX 3m put where I buy 100k vega for each day that the SPX is above 6500 upto a max of 1m vega, exit portfolio if SPX below 6500” 


In [ ]:
# Define backtest dates
start_date = dt.date(2024, 9, 1)
end_date = dt.date(2025, 9, 25)

In [ ]:
# Get data set for SPX spot
ds = Dataset('TREOD')
data = ds.get_data(start_date, assetId=['MA4B66MW5E27U8P32SB'])
s = pd.Series(data['closePrice'].to_dict())
data_source = GenericDataSource(s, MissingDataStrategy.fill_forward)

In [ ]:
opt = EqOption(underlier='.VIX', expiration_date='3m', option_type='put')

In [ ]:
# Define trade to add
trade_action_scaled = AddScaledTradeAction(
    priceables=opt,
    trade_duration='3m',
    scaling_type=ScalingActionType.risk_measure,
    scaling_risk=EqVega,
    scaling_level=100000,
    name='EnterAction',
)

# Define a trigger which will trigger if the SPX is above 6500 and the number of trades is less than 10

spx_above_6500 = MktTriggerRequirements(data_source, 6500, TriggerDirection.ABOVE)
trade_count_below_10 = TradeCountTriggerRequirements(10, TriggerDirection.BELOW)
enter_trigger = AggregateTrigger(
    AggregateTriggerRequirements([spx_above_6500, trade_count_below_10], AggType.ALL_OF), trade_action_scaled
)

# Define action to sell all instruments
exit_action = ExitAllPositionsAction(name='ExitAction')

# Define a trigger which will trigger if the SPX is below 6500
exit_trigger = MktTrigger(MktTriggerRequirements(data_source, 6500, TriggerDirection.BELOW), exit_action)

strategy = Strategy(None, [enter_trigger, exit_trigger])

In [ ]:
# Run backtest daily
GE = GenericEngine()
backtest = GE.run_backtest(strategy, start=start_date, end=end_date, frequency='1b', show_progress=True)

In [ ]:
backtest.trade_ledger()

In [ ]:
s.plot()